In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
    
MODEL = os.getenv('MODEL')
BASE_URL = os.getenv('BASE_URL')
API_KEY=os.getenv('GOOGLE_API_KEY')

openai = OpenAI(base_url=BASE_URL, api_key=API_KEY)

In [ ]:
system_prompt = "You are an help AI assistant for a well known Airlines company called AIRBUS.Help users by giving accurate information. \
    If you don't have answer ask them to vist website for accurate information"

In [ ]:
import sqlite3

In [ ]:
load_dotenv(override=True)
DB=os.getenv('DB')

with sqlite3.connect(DB) as conn:
    cursor=conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"Getting ticket price for city:{city}",flush=True)
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.execute('SELECT price from prices where city = ?',(city.lower(),))
        result=cursor.fetchone()
        return f"The ticket fare for {city} is {result[0]}" if result else f"No ticket data available for {city}"


In [ ]:
get_ticket_price('hyderabad')

In [ ]:
def set_ticket_price(city,price):
    print(f"Setting ticket price for city:{city} as price:{price}",flush=True)
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(),price,price))
        conn.commit()
    

In [ ]:
set_ticket_price('London',500)

In [ ]:
get_ticket_price('berlin')

In [ ]:
get_price_function={
    "name":"get_ticket_price",
    "description":"Gets ticket price for a city",
    "parameters":{
        "type":"object",
        "properties":{
            "city":{
                "type":"string",
                "description":"Name of the city that user wants to travel to"
            }
        },
        "required":["city"],
        "additionalProperties": False
    }
}

In [ ]:
set_price_function={
    "name":"set_ticket_price",
    "description":"Sets ticket price for a city in datastore",
    "parameters":{
        "type":"object",
        "properties":{
            "city":{
                "type":"string",
                "description":"Name of the city that user wants to set ticket price for"
            },
            "price":{
                "type":"number",
                "description":"Ticket price for the city"
            }
        },
        "required":["city","price"],
        "additionalProperties": False
    }
}

In [ ]:
tools=[{"type":"function","function":get_price_function},{"type":"function","function":set_price_function}]

In [ ]:
tools

In [ ]:
def chat(message,history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=openai.chat.completions.create(model=MODEL,messages=messages,tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        tool_responses=handle_tool_calls(message)
        messages.append(message)
        messages.append(tool_responses)
        response=openai.chat.completions.create(model=MODEL,messages=messages,tools=tools)
    
    return response.choices[0].message.content


In [ ]:
def handle_tool_calls(message):
    tools_response=[]
    def handle_get_ticket_price(args):
       city=arguments.get('city')
       return get_ticket_price(city)
    def handle_set_ticket_price(args):
        city=arguments.get('city')
        price=arguments.get('price')
        return f"Ticket price for city:{city} has been set successfully as {price}"
    
    tool_handlers={
        "get_ticket_price": handle_get_ticket_price,
        "set_ticket_price": handle_set_ticket_price
    }
    
    for tool_call in message.tool_calls:
        function_name=tool_call.function.name
        arguments=json.loads(tool_call.function.arguments)

        tools_response.append({
            "role": "tool",
            "content" : tool_handlers.get(function_name)(arguments),
            "tool_call_id" : tool_call.id
        })
        
    return tools_response


In [ ]:
gr.ChatInterface(fn=chat,type="messages").launch()